# Notebook 24 — Comprehensive model-search decision dashboard

**Goal:** Stop iterating blindly. Run a substantial, predefined comparison across feature families and classifier families, then evaluate **two separately selected policies** on three untouched chronological development folds:

- **Accuracy policy:** Highest inner-fold overall accuracy (balanced accuracy tie-breaker).
- **Entry policy:** Highest inner-fold **class-1 precision lift** subject to ≥5% coverage **in every inner fold**, ≥100 pooled entries, and positive lift. Abstain if no rule qualifies.

Target: seven training-fitted quantiles mapped `[2,2,0,0,1,1,1]` (2=down, 0=middle, 1=up). Balanced training is the default. The unweighted logistic candidate is explicitly diagnostic. **Never select a winner based on the outer evaluation table.**

The historical final holdout is closed. The prospective prediction ledger is not accessed. Trading returns are simplified next-bar proxies, not executable portfolio P&L.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Run from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.comprehensive_search import run_comprehensive_search,CANDIDATES,THRESHOLDS
print('Repository:',ROOT)

## 1. Predeclared experiment design

In [ ]:
N_OUTER=3
N_INNER=2
MAX_TRAIN_ROWS=50000
TREE_TRAIN_ROWS=35000
MIN_COVERAGE=.05
MIN_ENTRIES=100
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'comprehensive_search'/'1d'
print('Candidates:',len(CANDIDATES))
display(__import__('pandas').DataFrame([c.__dict__ for c in CANDIDATES]))
print('Probability thresholds:',THRESHOLDS)
print('Expected inner fits:',N_OUTER*N_INNER*len(CANDIDATES),'(+ selected outer fits)')

## 2. Load existing master table (no live data ingestion)

In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
found=next((p for p in paths if all((p/f).exists() for f in required)),None)
if found is None: raise FileNotFoundError('Run Notebook 03 to create master artifacts first')
master, latest, bundle, manifest = load_master(found)
print('Master rows:',len(master),'Master directory:',found)

## 3. Run nested comprehensive search (long-running)

In [ ]:
inner,outer,choices,cutoff=run_comprehensive_search(master,n_outer=N_OUTER,n_inner=N_INNER,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,
    min_coverage=MIN_COVERAGE,min_entries=MIN_ENTRIES,cost_bps=COST_BPS,output_dir=OUTPUT)
print('Development cutoff:',cutoff)
print('Inner grid rows:',len(inner),'Outer policy rows:',len(outer))
display(choices)

## 4. Decision dashboard — independent outer-fold evaluation

In [ ]:
cols=['outer_fold','policy','candidate','accuracy','majority_baseline_accuracy','balanced_accuracy','roc_auc',
      'up_precision','up_recall','coverage','precision_up','precision_lift','mean_entry_net_return','always_up_net_return']
display(outer[[c for c in cols if c in outer.columns]])
summary=outer.groupby('policy').agg(folds=('outer_fold','count'),accuracy=('accuracy','mean'),
                                    roc_auc=('roc_auc','mean'),balanced_accuracy=('balanced_accuracy','mean'),
                                    baseline_accuracy=('majority_baseline_accuracy','mean'),
                                    entry_precision=('precision_up','mean'),coverage=('coverage','mean'),
                                    net_return=('mean_entry_net_return','mean')).reset_index()
display(summary)
print('Important: NaNs in entry-policy metrics mean abstention or unavailable observations, not zero returns.')

## 5. Inner-fold candidate diagnostics (not an unbiased performance estimate)

In [ ]:
rank=inner.groupby('candidate',as_index=False).agg(
    mean_accuracy=('accuracy','mean'),mean_auc=('roc_auc','mean'),mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_fit_seconds=('fit_seconds','mean')).sort_values('mean_accuracy',ascending=False)
display(rank)
print('These rankings are for explaining inner selections, not for choosing a model from outer outcomes.')

## 6. Class-1 threshold / coverage tradeoff (inner folds only)

In [ ]:
thresholds=inner.groupby(['candidate','threshold'],as_index=False).agg(
    n=('n','sum'),entries=('entries','sum'),wins=('up_wins','sum'),base_up=('up_base_wins','sum'))
thresholds['coverage']=thresholds.entries/thresholds.n
thresholds['precision']=thresholds.wins/thresholds.entries.replace(0,float('nan'))
thresholds['precision_lift']=thresholds.precision-thresholds.base_up/thresholds.n
display(thresholds.sort_values('precision_lift',ascending=False).head(25))

## 7. Readout and next decision

Interpret **both** policies across the three outer folds. If neither beats its corresponding baseline consistently, do not promote a model. If the entry policy abstains, that is a valid safety outcome. Do not adjust candidate lists or thresholds based on outer scores; any future revisions require a new development experiment and fresh prospective validation.

**Next possible work:** assess per-sector and per-volatility-regime performance, explicit tradability and liquidity filters, and news sentiment features once a point-in-time historical archive is available in `stock_analytics_v0`.